# Classical Fourth-Order Runge–Kutta Method：完整 Taylor 对照推导

为把 RK4 的核心讲清楚，先暂时省略显式时间依赖，写成自治 ODE。Flow Matching 的一般情形是 f(x,t)=vθ(x,t)；将 t 看作状态向量的额外分量后，完全相同的 Runge–Kutta 推导成立。

$$\frac{dx}{dt}=f(x),\qquad x(t_n)=x_n.$$

目标是在不计算神经网络 Jacobian、Hessian 的前提下，以有限次 f(x) 调用近似 x(tₙ+h)。每次 f 调用对应一次 decoder NFE。

## 1. 精确积分的四阶 Taylor 展开到底是什么？

记 f′、f″、f‴ 分别为 f 对 x 的一阶、二阶、三阶导数；f″(a,b) 表示双线性 Hessian 作用，f′²f 表示 f′(f′(f))。沿精确轨迹反复求导：

$$x'=f,$$

$$x''=f'f,$$

$$x'''=f''(f,f)+f'^2f,$$

$$x''''=f'''(f,f,f)+3f''(f,f'f)+f'f''(f,f)+f'^3f.$$

所以精确解是

$$x(t_n+h)=x_n+h f+\frac{h^2}{2}f'f+\frac{h^3}{6}[f''(f,f)+f'^2f]$$

$$+\frac{h^4}{24}[f'''(f,f,f)+3f''(f,f'f)+f'f''(f,f)+f'^3f]+O(h^5).$$

四阶方法的含义就是：数值更新必须让上面 h、h²、h³、h⁴ 的每一个独立导数结构都具有相同系数。

## 2. 将四个 stage 也展开

RK4 选择：

$$k_1=f(x_n),$$

$$k_2=f(x_n+\frac{h}{2}k_1),$$

$$k_3=f(x_n+\frac{h}{2}k_2),$$

$$k_4=f(x_n+h k_3).$$

对这些 f(x+δ) 再作 Taylor 展开，保留到 h³（因为最终外面还会乘一个 h）：

$$k_1=f,$$

$$k_2=f+\frac{h}{2}f'f+\frac{h^2}{8}f''(f,f)+\frac{h^3}{48}f'''(f,f,f)+O(h^4),$$

$$k_3=f+\frac{h}{2}f'f+h^2[\frac14f'^2f+\frac18f''(f,f)]$$

$$+h^3[\frac1{16}f'f''(f,f)+\frac18f''(f,f'f)+\frac1{48}f'''(f,f,f)]+O(h^4),$$

$$k_4=f+h f'f+h^2[\frac12f'^2f+\frac12f''(f,f)]$$

$$+h^3[\frac14f'^3f+\frac18f'f''(f,f)+\frac12f''(f,f'f)+\frac16f'''(f,f,f)]+O(h^4).$$

这正解释了两个中点 stage 为什么不同：k₃ 含有由 k₂ 带来的 f′²f 等曲率项，而 k₂ 没有。

## 3. 逐项对照：1:2:2:1 从哪里来？

取加权平均 g=(k₁+2k₂+2k₃+k₄)/6。将上页四个展开式相加：

$$g=f+\frac{h}{2}f'f+\frac{h^2}{6}[f''(f,f)+f'^2f]$$

$$+\frac{h^3}{24}[f'''(f,f,f)+3f''(f,f'f)+f'f''(f,f)+f'^3f]+O(h^4).$$

数值更新为 xₙ₊₁=xₙ+h g，因此它与上一页的精确 Taylor 展开逐项相同，直到 h⁴：

$$x_{n+1}=x_n+h f+\frac{h^2}{2}f'f+\frac{h^3}{6}[f''(f,f)+f'^2f]$$

$$+\frac{h^4}{24}[f'''(f,f,f)+3f''(f,f'f)+f'f''(f,f)+f'^3f]+O(h^5).$$

所以 1:2:2:1 不是经验比例：它使加权 stage 展开的系数恰好成为 1/2、1/6、1/24。任意改变这些权重，至少一个 Taylor 项就无法匹配，四阶精度随之丢失。局部误差是 O(h⁵)，在固定区间累计后的全局误差为 O(h⁴)。

## 4. Flow Matching 的非自治时间和计算代价

真实模型为 f(x,t)。RK4 同时把 stage 时间放在左端、中点、中点、右端：

$$k_1=f(x_n,t_n),$$

$$k_2=f(x_n+hk_1/2,t_n+h/2),$$

$$k_3=f(x_n+hk_2/2,t_n+h/2),$$

$$k_4=f(x_n+hk_3,t_n+h).$$

$$x_{n+1}=x_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4).$$

其严格推导等价于把扩展状态 z=(x,t) 写成 dz/ds=(f(x,t),1)，再对 z 使用上一节自治 RK4。每个区间 4 NFE：

$$\mathrm{NFE}=4\times\mathrm{steps}. $$

若 encoder 输出缓存，则 encoder 只跑一次；每个 NFE 都是 decoder 一次。RK4(2 steps)=8 NFE，应与 Heun(4 steps)=8 NFE 或 Euler(8 steps)=8 NFE 比较。

In [ ]:
import torch


class ClassicalFourthOrderRungeKuttaMethod:
    """Classical RK4 for dx/dt = velocity(x, t); four decoder calls per interval."""

    def __init__(self, velocity):
        self.velocity = velocity
        self.nfe = 0

    @torch.no_grad()
    def sample(self, x_init, times):
        x = x_init
        self.nfe = 0
        for t_now, t_next in zip(times[:-1], times[1:]):
            step = t_next - t_now
            k1 = self.velocity(x, t_now)
            k2 = self.velocity(x + 0.5 * step * k1, t_now + 0.5 * step)
            k3 = self.velocity(x + 0.5 * step * k2, t_now + 0.5 * step)
            k4 = self.velocity(x + step * k3, t_next)
            # Weights 1:2:2:1 match the exact Taylor expansion through h^4.
            x = x + step * (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
            self.nfe += 4
        return x


times = torch.linspace(0.0, 1.0, 3)
constant = ClassicalFourthOrderRungeKuttaMethod(lambda x, t: torch.full_like(x, 2.0))
torch.testing.assert_close(constant.sample(torch.zeros(2, 2), times), torch.full((2, 2), 2.0))
assert constant.nfe == 8

# A non-constant autonomous test: exact x(1)=e for x'=x and x(0)=1.
linear = ClassicalFourthOrderRungeKuttaMethod(lambda x, t: x)
estimate = linear.sample(torch.ones(1, 1), times)
print("RK4 estimate:", estimate.item(), "exact e:", torch.exp(torch.tensor(1.0)).item())



## 实现与推导的逐项对应

代码中 `k1`、`k2`、`k3`、`k4` 分别是前文 Taylor 展开中的四个 stage；注意每个 $k_i$ 都是速度，不含步长 $h$。因此最终更新必须是

$$
x_{n+1}=x_n+h\frac{k_1+2k_2+2k_3+k_4}{6}.
$$

若把 $h$ 预先乘进 `k_i`，公式会是另一种记号，不能混用。常数速度测试验证权重和为一；$x'=x$ 测试给出可与 $e$ 对照的非平凡例子。真实 Flow Matching 是非自治方程，代码通过在四个 stage 显式传入 $t_n,t_n+h/2,t_n+h/2,t_n+h$ 来实现增广状态推导。
